# Chapitre 12 - Cours 4 : justesse de bout en bout et contraintes

Un service sans état qui se trompe se corrige en redémarrant ; un système qui
**mémorise** (une base de données) garde ses erreurs, parfois pour toujours. Ce notebook
ouvre la section *Aiming for Correctness* : il couvre *The End-to-End Argument for
Databases* (exécution exactement une fois, suppression des doublons, identifiants de
requête, argument de bout en bout) et *Enforcing Constraints* (unicité, journal
partitionné, traitement d'une requête qui touche plusieurs partitions sans validation atomique).

**Plan.**

1. Pourquoi la justesse est difficile.
2. Exécution exactement une fois et suppression des doublons.
3. Identifier chaque requête de bout en bout.
4. L'argument de bout en bout.
5. Faire respecter l'unicité.
6. Une requête sur plusieurs partitions, sans validation atomique.

In [1]:
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ddia import dataflow as dfl
from ddia.diagrams import Diagram
from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET, COLORS,
                           annotate_bars, finish, set_style, subtitle)
from ddia.timelines import SpaceTime

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 3)
SHARD, SHARDS = "partition", "partitions"

## 1. Pourquoi la justesse est difficile (section *Aiming for Correctness*)

On veut des applications **fiables et correctes** : des programmes dont la sémantique est
bien définie et comprise, même en présence de pannes. Depuis une quarantaine d'années,
l'outil de référence est la transaction (atomicité, isolation, durabilité). Ses
fondations sont pourtant plus fragiles qu'il n'y paraît : la confusion autour des
niveaux d'isolation faibles (chapitre 7) en témoigne ; certains domaines
ont abandonné les transactions pour des modèles plus performants mais à la sémantique
floue ; la « cohérence » est souvent invoquée et rarement définie. Les expériences Jepsen
de Kyle Kingsbury ont montré de grands écarts entre les garanties annoncées par certains
produits et leur comportement réel face aux pannes réseau et aux plantages. Et même avec
une base parfaite, l'application doit encore utiliser correctement des réglages
difficiles à comprendre (isolation, quorums).

Une base qui offre des transactions sérialisables ne protège pas d'un **bogue
applicatif** qui écrit des données fausses ou en supprime. C'est un argument en faveur
des données **immuables, en ajout seul** : on se remet bien plus facilement d'une erreur
si le code fautif n'a pas pu **détruire** les bonnes données. Comparons, après un
déploiement bogué qui applique une remise deux fois trop forte pendant une heure, une
table de soldes **mise à jour en place** et un **journal d'événements** dont on dérive les
soldes.

In [2]:
rng = random.Random(4)
events = []                                                 # (heure, client, montant brut, remise)
for h in range(24):
    for _ in range(200):
        events.append((h, rng.randrange(500), round(rng.uniform(5, 200), 2), 0.10))

def montant(evt, bug):
    h, c, brut, remise = evt
    r = remise * 2 if (bug and h == 13) else remise          # le déploiement bogué de 13 h
    return brut * (1 - r)

soldes_en_place = {}
for e in events:                                            # mise à jour en place : seul le résultat survit
    soldes_en_place[e[1]] = soldes_en_place.get(e[1], 0) + montant(e, bug=True)
soldes_corrects = {}
for e in events:                                            # journal : on peut tout rejouer avec le code corrigé
    soldes_corrects[e[1]] = soldes_corrects.get(e[1], 0) + montant(e, bug=False)
faux = sum(abs(soldes_en_place[c] - soldes_corrects[c]) > 1e-9 for c in soldes_corrects)
pd.DataFrame({"approche": ["table mise à jour en place", "journal immuable + rejeu corrigé"],
              "soldes faux après correction du bogue": [faux, 0],
              "comment réparer ?": ["impossible sans l'historique (montants bruts perdus)",
                                    "rejouer les événements avec le code corrigé"]})

,approche,soldes faux après correction du bogue,comment réparer ?
0,table mise à jour en place,160,impossible sans l'historique (montants bruts p...
1,journal immuable + rejeu corrigé,0,rejouer les événements avec le code corrigé


La table en place a « oublié » les montants bruts : les soldes de plusieurs centaines de
clients restent faux, et rien ne permet de savoir de combien. Le journal immuable
conserve l'entrée de chaque décision ; corriger le code et rejouer suffit. L'immuabilité
n'est pas une panacée pour autant : la suite montre une corruption plus subtile, qui
survit à des transactions parfaitement sérialisables.

> **À retenir.** Les garanties d'une base (même sérialisable) ne protègent ni des bogues
> applicatifs ni des mauvais usages. Garder les données immuables, en ajout seul, empêche
> le code fautif de détruire les bonnes données et rend la réparation possible.